In [1]:
from dance.transforms import Compose
from dance.datasets.spatial import SpatialLIBDDataset
from dance.modules.spatial.spatial_domain.spagcn import SpaGCN
from dance.utils import set_seed
from dance.transforms import CellPCA, Compose, FilterGenesMatch, SetConfig
from dance.transforms.graph import SpaGCNGraph, SpaGCNGraph2D

/mnt/g/Projects/dance/dance/utils/matrix.py:178: NumbaExperimentalFeatureWarning: First-class function type feature is experimental
  for j in numba.prange(n):


In [2]:
import scanpy as sc
import numpy as np
import gc
import pandas as pd

In [3]:
from dance.modules.spatial.spatial_domain.stagate import Stagate

In [4]:
import sklearn.metrics

In [5]:
from tqdm import tqdm

In [6]:
from dance import logger
logger.setLevel("ERROR")

In [7]:
import time


In [8]:
from dance.transforms import AnnDataTransform, Compose, SetConfig
from dance.transforms.graph import StagateGraph
from dance.typing import Any, LogLevel, Optional, Tuple

def preprocessing_pipeline(model_name: str = "knn", radius: float = 150, n_neighbors: int = 8, 
                           log_level: LogLevel = "ERROR"):
        return Compose(
            AnnDataTransform(sc.pp.normalize_total, target_sum=1e4),
            AnnDataTransform(sc.pp.log1p),
            StagateGraph(model_name, radius=radius, n_neighbors=n_neighbors),
            SetConfig({
                "feature_channel": "StagateGraph",
                "feature_channel_type": "obsp",
                "label_channel": "label",
                "label_channel_type": "obs"
            }),
            log_level=log_level,
        )

preprocessing_pipeline = preprocessing_pipeline()

In [12]:
from dance.datasets.base import Data
aris = []
nmis = []
t0 = time.time()
adata = sc.read_h5ad("../../../data/slidetags/HumanTonsil_2000.h5ad")
pixel_factor = 80 / ((adata.obsm['spatial'][:, 0].max() - adata.obsm['spatial'][:, 0].min()) * 
       (adata.obsm['spatial'][:, 1].max() - adata.obsm['spatial'][:, 1].min()) 
       / adata.shape[0]) ** .5
adata.obsm['spatial_pixel'] = adata.obsm['spatial'] * pixel_factor 
adata.obs['label'] = 0

data = Data(adata)
preprocessing_pipeline(data)

adj, y = data.get_data(return_type="default")
x = data.data.X
edge_list_array = np.vstack(np.nonzero(adj))

model = Stagate(hidden_dims=[x.shape[1], 512, 32])
pred = model.fit_predict((x, edge_list_array), 1000, num_cluster=3, random_state=0)
adata.obs['stagate'] = pred
adata.obs[['stagate']].to_csv(f"../../Steamboat/revised/saved_results/tonsil_stagate_spatial_domain.csv")

# ari = sklearn.metrics.adjusted_rand_score(adata.obs['parcellation_division'], adata.obs['stagate'])
# nmi = sklearn.metrics.normalized_mutual_info_score(adata.obs['parcellation_division'], adata.obs['stagate'])

# print(i, adata.obs['stagate'].nunique(), ari, nmi, f"{(time.time() - t0) / 60:.2f} minutes")
# aris.append(ari)
# nmis.append(nmi)

# del adata
# gc.collect()

tt = time.time() - t0

100%|███████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:10<00:00, 94.37it/s]


In [13]:
adata.obs['stagate'].value_counts()

stagate
1    2391
2    2352
0    1035
Name: count, dtype: int64